# (1+1)D Korteweg–De Vries (KdV) Equation with Physics-Informed Neural Networks

The **KdV equation** models the dynamics of solitary waves (solitons):

$$u_t + \eta\, u\, u_x + \mu^2\, u_{xxx} = 0, \quad t \in (0, 1),\ x \in (-1, 1)$$

**Initial condition:**
$$u(x, 0) = \cos(\pi x)$$

**Periodic boundary conditions:**
$$u(t, -1) = u(t, 1)$$

**Classical parameters** (η = 1, μ = 0.022).


In [ ]:
import pinns

import numpy as np
import jax.numpy as jnp

## Domain
Space-time domain: $x \in [-1, 1]$, $t \in [0, 1]$. Periodic BC on $x$.


In [2]:
domain = pinns.DomainCubic(
    space=[(-1.0, 1.0)],   # x
    time =[0.0,   1.0],    # t
)

domain.add_periodic("x", "periodic")


## PDE Residual

$$\mathcal{R} = u_t + \eta\, u\, u_x + \mu^2\, u_{xxx} = 0$$


In [ ]:
problem = pinns.ProblemStrong(domain=domain, output_names=["u"])

def kdv_residual(X, U, params, derivative=None):
    eta = params['parameter']['eta']
    mu2 = params['parameter']['mu2']

    u_t   = derivative(U, X, 0, (1,))           # ∂u/∂t
    u_x   = derivative(U, X, 0, (0,))           # ∂u/∂x
    u_xxx = derivative(U, X, 0, (0, 0, 0))      # ∂³u/∂x³

    return u_t + eta * U * u_x + mu2 * u_xxx    # (N,1)

problem.add_inner(kdv_residual, name="pde")

def u_initial(X):
    x = X[:, 0:1]
    return jnp.cos(jnp.pi * x)

problem.add_initial(u_initial, name="initial")

problem.add_parameter("eta", 1.0)
problem.add_parameter("mu2", 0.022**2)


ProblemStrong(domain=DomainCubic, n_dims=2, n_outputs=1, inner=1, boundary=0, initial=1, fixed=['eta', 'mu2'], inferred=[])

### Reference Solution (spectral ETD2RK)

The KdV equation with periodic BCs is solved via `DomainCubic` + `ModelSpectralSolver`.
In Fourier space:

$$\partial_t \hat{u}_k = \underbrace{i\mu^2 k^3 \hat{u}_k}_{\text{linear — exact}} + \underbrace{-\frac{i\eta k}{2}\widehat{u^2}_k}_{\text{nonlinear — ETD2RK}}$$

The linear dispersion term is integrated exactly; `IntegratorETD2RK` uses `jax.lax.scan` and is fully differentiable for inverse problems.


In [ ]:
import jax
eta_val = 1.0
mu_val  = 0.022
Nx = 512
Nt = 200

t_ref = np.linspace(0, 1, Nt)

# ── DomainCubic ────────────────────────────────────
domain_grid = pinns.DomainCubic(
    space=[(-1.0, 1.0)],
    time=(0.0, 1.0),
)
x_ref = np.array(domain_grid.x)   # (512,) periodic nodes

# ── ModelSpectralSolver (integrator owned by the model) ──────────────────────────────
# Operators use a flat p dict: p["mu2"], p["eta"], etc.
integrator = pinns.IntegratorETD2RK(dt=5e-4)
model_solver = pinns.ModelSpectralSolver(domain_grid, ["u"], integrator)

# Linear part:  L_k = i·μ²·k³   (k · K2 = k · k² = k³)
model_solver.set_linear_op(
    lambda K2, p: {"u": 1j * p["mu2"] * domain_grid.k * K2}
)

# Nonlinear part:  N(û)_k = -i·η·k/2 · FFT(u²)
def kdv_nonlinear(state_hat, p):
    u = domain_grid.inverse(state_hat["u"])
    return {"u": -1j * p["eta"] / 2.0 * domain_grid.k
                 * domain_grid.forward(u * u)}

model_solver.set_nonlinear_op(kdv_nonlinear)
model_solver.add_parameter("eta", eta_val)
model_solver.add_parameter("mu2", mu_val**2)

# Initial condition  u(x, 0) = cos(πx)
model_solver.add_initial(jnp.cos(jnp.pi * domain_grid.x))

# ── Forward solve  (dt=5e-4 → 2000 steps) ───────────────────────────────────
U_etd = model_solver.solve(t_obs=t_ref)

ProblemStrong(domain=DomainCubic, n_dims=2, n_outputs=1, inner=1, boundary=0, initial=1, fixed=['eta', 'mu2'], inferred=[])

## Network Architecture

Fourier feature embedding + FNN with `tanh` output activation to keep predictions bounded.
The KdV solution stays within the amplitude range $\approx [-1.5, 1.5]$.


In [5]:
network = pinns.ModelBase(domain, 1)

network.add(
    pinns.PeriodicEmbedding("x")
)
network.add(
    pinns.FourierFeatures(256, sigma=1.0)
)
network.add(
    pinns.PirateNet(256, 3, rwf_mu=1, rwf_sigma=0.1)
)

ModelBase(output_dim=1)
  [PeriodicEmbedding_0] PeriodicEmbedding([axis=0 freq=3.1416 k=[1]], input_dim=2, output_dim=3)
  [RandomFourierFeatures_1] RandomFourierFeatures(fourier_input_dim=3, n_features=256, sigma=1.0, adaptive=True)
  [PirateNet_2] PirateNet(input_dim=512, hidden_dim=256, n_blocks=3, output_dim=1)

## Training

Three schedulers mirror the jaxpi KdV setup:

- **Causal weighting** (`SchedulerCausal`): time-sorts the PDE collocation points and exponentially down-weights future chunks until earlier chunks converge ($\varepsilon = 1.0$, 16 chunks).
- **Grad-norm balancing** (`SchedulerGradNorm`): adapts per-term loss weights every 1 000 epochs so that their gradient magnitudes stay proportional.

- **Curriculum + resample**: the sampled time window grows from $t\in[0,0.1]$ to $[0,1]$ over 10 stages of 5 000 epochs each; collocation points are resampled every 1 000 epochs.

In [ ]:
trainer = pinns.Trainer(network, problem=problem)

trainer.compile(
    problem = {
        "pde":      {"train": 1000, "test": 10, "weight": 1.0},
        "initial":  {"train": 200,  "test": 10, "weight": 1.0},
    },
    optimizer=pinns.SOAPOptimizer(),
    schedulers=[
        pinns.SchedulerResample(1),
        pinns.SchedulerCausal(term="pde", tol=1.0, n_chunks=16, t_col=1),
    ],
    epochs     = 500000,
    print_each = 1000,
)

trainer.train()